# Fábrica de GIFs — Fire Collection 5

Notebook interativo para gerar GIFs animados de dados de fogo do MapBiomas (Coleção 5) via Google Earth Engine.

**Funciona em:**
- Google Colab (Pro: 8 workers, RAM ampliada)
- VS Code (Jupyter extension, usa venv local)

---

In [ ]:
# @title Célula 1: Setup & Ambiente

import sys, os, multiprocessing

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    !pip install -q earthengine-api pillow pyyaml ipywidgets 2>/dev/null
    !git clone -q https://github.com/ipam/fabrica-de-gifs.git 2>/dev/null || true
    %cd fabrica-de-gifs
    import ee
    ee.Authenticate()
    ee.Initialize(project='ee-ipam')
else:
    cwd = os.getcwd()
    sys.path.insert(0, cwd)
    if not os.path.exists(os.path.join(cwd, 'src')):
        parent = os.path.dirname(cwd)
        if os.path.exists(os.path.join(parent, 'src')):
            sys.path.insert(0, parent)
    import ee
    try:
        ee.Initialize(project='ee-ipam')
    except Exception:
        pass

from src.ipam_gif_factory.config import ConfigLoader
from src.ipam_gif_factory.core.pipeline import Pipeline

config = ConfigLoader()

WORKERS = max(1, min((os.cpu_count() or 4) - 1, 12))

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'VS Code'}")
print(f"Workers auto-detectados: {WORKERS}")
print(f"Datasets disponíveis: {len(list(config.datasets.keys()))}")
print(f"Territórios disponíveis: {len(list(config.territories.keys()))}")
print("Pronto!")

In [ ]:
# @title Célula 2: Seleção de Dataset, Produtos e Territórios

import ipywidgets as widgets
from IPython.display import display, clear_output

dataset_dd = widgets.Dropdown(
    options=list(config.datasets.keys()),
    value='brasil_fire_col5',
    description='Dataset:'
)

product_ms = widgets.SelectMultiple(
    options=[],
    description='Produtos:',
    rows=10,
    layout=widgets.Layout(width='400px')
)

territory_ms = widgets.SelectMultiple(
    options=sorted(config.territories.keys()),
    value=['df', 'brasil', 'amazonia', 'cerrado'],
    description='Territórios:',
    rows=15,
    layout=widgets.Layout(width='300px')
)

workers_tx = widgets.IntText(value=WORKERS, description='Workers:')
resume_cb = widgets.Checkbox(value=True, description='Resume')
collage_cb = widgets.Checkbox(value=True, description='Collage')
dimension_tx = widgets.IntText(value=1560, description='Altura px:')

run_btn = widgets.Button(description='Gerar Batch', button_style='primary')
output_area = widgets.Output()

def on_dataset_change(change):
    ds_id = change['new']
    if ds_id in config.datasets:
        prods = list(config.datasets[ds_id].get('products', {}).keys())
        product_ms.options = prods
        if prods:
            product_ms.value = (prods[0],)

dataset_dd.observe(on_dataset_change, names='value')
on_dataset_change({'new': dataset_dd.value})

display(widgets.HBox([dataset_dd]))
display(widgets.HBox([product_ms, territory_ms]))
display(widgets.HBox([workers_tx, resume_cb, collage_cb, dimension_tx]))
display(run_btn)
display(output_area)

In [ ]:
# @title Célula 3: Execução do Batch

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

print_lock = threading.Lock()
results_lock = threading.Lock()
results_list = []

def process_one(dataset_id, prod, territory, resume):
    pipeline = Pipeline(config)
    result = pipeline.run(
        dataset_id=dataset_id,
        product_id=prod,
        territory_id=territory,
        create_collage=collage_cb.value,
        add_labels=True,
        vertical_dimension=dimension_tx.value,
        cell_height=300,
        resume=resume,
    )
    with results_lock:
        results_list.append(result)
    return result

def on_run_click(b):
    with output_area:
        clear_output()
        
        dataset_id = dataset_dd.value
        products = list(product_ms.value)
        territories = list(territory_ms.value)
        workers = workers_tx.value
        resume = resume_cb.value
        
        if not products or not territories:
            print("Selecione pelo menos 1 produto e 1 território.")
            return
        
        combos = [(p, t) for t in territories for p in products]
        total = len(combos)
        results_list.clear()
        
        print(f"Dataset: {dataset_id}")
        print(f"Produtos: {len(products)}")
        print(f"Territórios: {len(territories)}")
        print(f"Total: {total} combinações")
        print(f"Workers: {workers} | Resume: {resume}")
        print(f"{'=' * 50}")
        
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(process_one, dataset_id, p, t, resume): (p, t)
                for p, t in combos
            }
            for i, f in enumerate(as_completed(futures), 1):
                p, t = futures[f]
                result = f.result()
                status = result.get('status', '?')
                print(f"[{i}/{total}] {p} / {t} → {status}")
        
        ok = sum(1 for r in results_list if r['status'] == 'success')
        fail = sum(1 for r in results_list if r['status'] == 'error')
        print(f"\n{'=' * 50}")
        print(f"RESUMO: {ok} OK / {fail} Falha / {total} Total")

run_btn.on_click(on_run_click)

In [ ]:
# @title Célula 4: Resultados & Preview

from IPython.display import Image as IPImage, display as ipydisplay
import json, glob

output_base = config.get_output_dir()

if results_list:
    print("## Últimos GIFs gerados:\n")
    for i, r in enumerate(results_list):
        if r['status'] == 'success':
            gif = r.get('gif_path', '')
            if gif and os.path.exists(gif):
                size_mb = os.path.getsize(gif) / (1024 * 1024)
                print(f"{i+1}. {r['product']} / {r['territory']} ({size_mb:.1f} MB)")

    print(f"\n## Preview dos 3 primeiros:\n")
    count = 0
    for r in results_list:
        if count >= 3:
            break
        if r['status'] == 'success' and r.get('gif_path') and os.path.exists(r['gif_path']):
            print(f"### {r['product']} — {r['territory']}")
            ipydisplay(IPImage(filename=r['gif_path']))
            count += 1
else:
    print("Nenhum resultado ainda. Rode a Célula 3 primeiro.")
    print(f"\nProcurando GIFs existentes em {output_base}...")
    gifs = sorted(glob.glob(f"{output_base}**/*.gif", recursive=True))
    print(f"Encontrados: {len(gifs)} GIFs")
    for g in gifs[:5]:
        print(f"  {os.path.relpath(g, output_base)}")